# Word2Vec：仅名词词形还原

本个人入口使用原项目训练集的 `words`，只在模型私有列中还原 POS 标注为 `N*` 的名词。保留 token 数量与顺序、数字、否定词及 `better/best/worse/worst` 等程度表达，不覆盖公共文本列；不使用 WebMD、短语或停用词过滤。词表与词向量只从 train 学习，valid/test 只应用冻结规则并生成向量。

| 参数 | 默认值 |
|---|---:|
| vector_size | 300 |
| window_size | 5 |
| min_count | 5 |
| max_iter | 10 |
| step_size | 0.025 |
| seed | 42 |
| num_partitions | 1 |
| max_sentence_length | 1000 |
| pooling | meanmax |

meanmax 拼接均值和逐坐标最大值，输出 **600维**；均值分母包括 OOV，OOV 不参与最大值。空评论或全 OOV 输出定长零向量。`p01` 为300维平均聚合，同样先做仅名词还原。

环境沿用仓库 README 的 Python 3.12 / PySpark 4.2 / Java 21。当前交付模型另需 **NLTK 3.9.3、WordNet 3.0** 及英文 POS 资源，在 driver 和 Python worker 环境执行一次：

```bash
python -m pip install nltk==3.9.3
python -m nltk.downloader averaged_perceptron_tagger_eng wordnet
```

资源使用 NLTK 默认搜索路径或 `NLTK_DATA`，代码不自动下载。模型保存资源指纹，预测时核对版本与文件 SHA256。Java/Hadoop 在外部配置，本 notebook 不写入个人机器路径。

## 1. 运行配置与会话

- `REBUILD=False`：读取已有公共 split。首次运行无缓存，或公共清洗、文本特征、划分、编码改变时，按团队约定设为 `True` 重建；这会覆盖公共 split。
- `TRAIN=False`：默认加载本次交付的仅名词还原模型并重新生成向量缓存。`TRAIN=True` 在 train 上重新拟合，按 `uniqueID` 升序组织训练输入，替换 `model_name` 对应模型。`REBUILD` 不控制模型训练。
- `word2vec_config={}`：使用模块默认值；自定义参数须同时设置 `TRAIN=True`。预测始终读取保存配置。
- 其他实验使用独立 `model_name`，避免覆盖正式模型。仅名词规则依赖 POS 和 WordNet，仍可能出现如 `doses→dos` 的歧义，没有加入数据集特例。

若加载的是旧原词形模型，此入口会明确报错；请更新交付模型，或用当前模块设置 `TRAIN=True` 重训。修改模块、Java或内存后重启 kernel。

In [ ]:
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI
from src.paths import EMBEDDING_MODELS_DIR, EMBEDDING_DIR, SPLIT_DIR

from src.seed import SEED
REBUILD = False
TRAIN = False
embedding_name = "word2vec"
model_name = "word2vec"
word2vec_config = {}

if not TRAIN and word2vec_config:
    raise ValueError("Custom config requires TRAIN=True; prediction uses the saved model config.")

spark = (SparkSession.builder
         .master("local[4]")
         .appName("Word2VecPipeline")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.parquet.columnarReaderBatchSize", "512")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
data_oi = DataOI()

## 2. 准备公共输入

沿用公共清洗、文本特征、seed=42 哈希划分及训练集拟合的编码，不另加去重、过滤或公共文本修复。Word2Vec 只读取 `words`，模型内部的名词还原不改变这些缓存。

Git pull 不会更新本机 Parquet。运行会记录实际输入行数和文件内容的聚合指纹，不能把历史实验分数当作最新公共代码上的重跑结果。

In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_cleaned, df_engineered)
    del df_raw, df_cleaned, df_engineered
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


## 3. 训练或读取模型

固定注册名 `word2vec`，由 `model_name` 指定模型目录。train 每次重新训练并保存参数；predict 从磁盘读取模型及配置，不在 valid/test 拟合。此入口要求保存配置声明 `noun_only`，防止静默复用旧原词形模型。

下方输入 SHA256 按相对文件名和文件内容依次汇总，不另保存数千行逐文件清单。本次代码提交、模式、输入来源和输出信息保存在本地生成缓存目录的 `run_record.json`，不提交 Git。

In [ ]:
from src.embeddings import EmbeddingModel

embedding_model = EmbeddingModel(df_train, df_valid, df_test)
if TRAIN:
    print(f"Training noun-only Word2Vec: {model_name}")
    EmbeddingModel.train_registry[embedding_name](
        embedding_model, model_name=model_name, config=word2vec_config,
    )
else:
    print(f"Reusing saved noun-only Word2Vec: {model_name}")

saved_config = json.loads(
    (EMBEDDING_MODELS_DIR / model_name / "embedding_config.json").read_text(encoding="utf-8")
)
if (saved_config.get("format_version") != 2
        or saved_config.get("normalization", {}).get("policy") != "noun_only"):
    raise ValueError("This notebook requires the noun-only model. Update the delivered model or set TRAIN=True with the current module.")
print(json.dumps(saved_config, ensure_ascii=False, indent=2))


def input_fingerprint(directory):
    digest = hashlib.sha256()
    files = sorted(directory.rglob("*.parquet"))
    for path in files:
        digest.update(path.relative_to(directory).as_posix().encode("utf-8") + b"\0")
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
    return {"path": str(directory), "parquet_files": len(files), "sha256": digest.hexdigest()}


run_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "code_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "embedding": model_name,
    "trained_this_run": TRAIN,
    "split_rebuilt": REBUILD,
    "split_seed": SEED,
    "saved_model_config": saved_config,
    "inputs": {
        name: {**input_fingerprint(SPLIT_DIR / f"{name}.parquet"), "rows": frame.count()}
        for name, frame in zip(["train", "valid", "test"], [df_train, df_valid, df_test])
    },
}

## 4. 生成并保存三份向量

predict 使用模型保存的还原规则和聚合配置，返回 train、valid、test。基类保留 ID、标签、训练权重及编码特征，`embedding` 为 Spark ML `VectorUDT`。

通过 `DataOI` 保存到 `data/processed/embeddings/<model_name>/`，会替换同名旧向量缓存。输入行数、文件指纹及输出位置一并留痕。下游分类器需要重新拟合；此 notebook 只负责 Word2Vec，不交叉执行其他成员的 analytics。

In [ ]:
print(f"Generating embeddings: {model_name}")
train_embedding, valid_embedding, test_embedding = (
    EmbeddingModel.predict_registry[embedding_name](embedding_model, model_name)
)
data_oi.save_embedding_data(
    model_name, train_embedding, valid_embedding, test_embedding,
)
run_record["output"] = {
    "path": str(EMBEDDING_DIR / model_name),
    "dimension": saved_config["output_dimension"],
    "rows": {
        name: frame.count()
        for name, frame in zip(["train", "valid", "test"], [train_embedding, valid_embedding, test_embedding])
    },
}
assert all(run_record["output"]["rows"][name] == record["rows"]
           for name, record in run_record["inputs"].items())
record_path = EMBEDDING_DIR / model_name / "run_record.json"
record_path.write_text(json.dumps(run_record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Saved {saved_config['output_dimension']}D embeddings to {EMBEDDING_DIR / model_name}")
print(f"Run record: {record_path}")
train_embedding.printSchema()
train_embedding.select("uniqueID", "rating", "embedding").show(3, truncate=80)

## 已选模型与历史实验记录

本次交付模型来自 `selectlemma_20261111_154957/noun_only`，模型名 `word2vec_noun_only_20261111_154957`；实验代码基线 `6e9ae80`，输入缓存来源提交 `46b1a03`。只在原训练集77,800条评论、6,702,823个token上训练；295,995个词次发生还原，minCount后词表13,812项。NLTK POS/WordNet是冻结语言资源，未使用验证或测试标签。

以下是同一历史划分、相同不加权 `lr_baseline` 配置下的验证结果，**不是本 notebook 在最新共享流程上的重跑成绩**：

| 方案 | 验证准确率 | Weighted F1 | MAE | RMSE |
|---|---:|---:|---:|---:|
| 仅名词还原（选中） | 40.4145% | 0.353624 | 1.740379 | 2.934713 |
| 原词形 | 40.2261% | 0.350837 | 1.770970 | 2.976644 |
| 全还原 | 39.5891% | 0.341728 | 1.833498 | 3.051485 |

真实4–7分验证样本共1,976条，正确数依次126、108、93。仅名词相对原词形净增加18条，探索性配对p约0.142；收益较小，未证明稳定优于原词形。三组LR均达到100次迭代上限，属于单seed、固定预算的比较；测试集只保存预测，未评分。

模型来源与输入摘要见模型目录的 `embedding_config.json`。此 notebook 每次实际运行的输入指纹与模式见本机向量缓存目录的 `run_record.json`。analytics 成员可用 `data_oi.load_embedding_data(model_name)` 读取三份向量，通过自己的个人入口训练和评分。

提交时清空代码单元输出和执行计数，不提交生成缓存或运行日志。